# Indusense — Sprint 1

Ce notebook produira les cartes d'identité des fichiers CSV placés dans `datas/`.

In [30]:
import csv
from codecs import BOM_UTF8
from collections.abc import Iterator
from datetime import date, datetime, time
from io import StringIO
from pathlib import Path

import pandas as pd
from sqlglot import exp, parse
from sqlglot.errors import ParseError, TokenError

from source_sql import DIALECTE_SQL, classer_colonne_sql, extraire_tables_sql

DATA_DIR = Path("datas")

## Lire les fichiers CSV

On conserve les valeurs brutes sous forme de texte pour préserver les identifiants. Les champs vides sont considérés comme manquants. L'encodage est déduit en essayant UTF-8, puis Windows-1252 et ISO-8859-1.

In [31]:
def lire_csv(chemin: Path) -> tuple[pd.DataFrame, str, str]:
    contenu = chemin.read_bytes()
    encodages = ("utf-8-sig",) if contenu.startswith(BOM_UTF8) else (
        "utf-8", "cp1252", "iso-8859-1"
    )
    for encodage in encodages:
        try:
            texte = contenu.decode(encodage)
        except UnicodeDecodeError:
            continue
        break
    else:
        raise ValueError(f"{chemin}: le fichier ne peut pas être décodé.")

    try:
        separateur = csv.Sniffer().sniff(texte[:8192], delimiters=",;\t").delimiter
    except csv.Error as erreur:
        raise ValueError(f"{chemin}: séparateur CSV indéterminé.") from erreur

    try:
        donnees = pd.read_csv(
            StringIO(texte),
            sep=separateur,
            dtype="string",
            keep_default_na=False,
            na_values=[""],
        )
    except (pd.errors.ParserError, pd.errors.EmptyDataError) as erreur:
        raise ValueError(f"{chemin}: lecture du CSV impossible ({erreur}).") from erreur

    return donnees, encodage, separateur

## Inférer les types et calculer les statistiques

Les colonnes connues suivent les règles explicites du projet. Pour les autres, toutes les valeurs présentes doivent être compatibles avec le type inféré. Les dates et heures inférées suivent les formats ISO. Les horodatages avec fuseau sont comparés en UTC. Une colonne mixte reste en texte et la carte indique la famille compatible avec le plus de valeurs. Les champs vides sont exclus de l’inférence et des statistiques.

In [32]:
IDENTIFIANTS = {"incident_id", "machine_id", "operator_badge"}
NUMERIQUES = {
    "releves_incidents.csv": {"severity"},
    "telemetry.csv": {
        "temperature_c", "pressure_bar", "voltage_mean_v",
        "rotation_mean_rpm", "pieces_produced",
    },
}
FORMATS_TEMPORELS = {
    "releves_incidents.csv": {
        "date": ("DATE", "%Y-%m-%d"),
        "time": ("TIME", "%H:%M"),
    },
    "telemetry.csv": {"timestamp": ("DATETIME", "%Y-%m-%d %H:%M:%S")},
}
COLONNES_INDICATEURS = {
    "type_surchauffe", "type_baisse_pression", "type_vibration",
    "type_bruit_mecanique", "type_surconsommation", "type_blocage_mecanique",
    "type_alarme_capteur", "type_arret_urgence", "type_defaut_qualite",
}
INDICATEURS = {"0": False, "1": True}
DATE_ISO = r"[0-9]{4}-[0-9]{2}-[0-9]{2}"
HEURE_ISO = r"(?:[01][0-9]|2[0-3]):[0-5][0-9](?::[0-5][0-9](?:\.[0-9]{1,6})?)?"
DATETIME_ISO = rf"{DATE_ISO}[T ]{HEURE_ISO}"
FUSEAU_ISO = r"(?:Z|[+-](?:[01][0-9]|2[0-3]):?[0-5][0-9])"


def convertir_numeriques(valeurs: pd.Series) -> pd.Series:
    """Écarte aussi les valeurs infinies des statistiques."""
    nombres = pd.to_numeric(valeurs, errors="coerce")
    return nombres.mask(nombres.isin([float("inf"), float("-inf")]))


def candidats_temporels(valeurs: pd.Series) -> Iterator[tuple[str, pd.Series]]:
    """Ne propose que des formats ISO non ambigus, sans mélanger les horloges."""
    for type_colonne, motif, avec_fuseau in (
        ("DATE", DATE_ISO, False),
        ("TIME", HEURE_ISO, False),
        ("DATETIME", DATETIME_ISO, False),
        ("DATETIME (avec fuseau)", DATETIME_ISO + FUSEAU_ISO, True),
    ):
        correspondantes = valeurs[valeurs.str.fullmatch(motif)]
        if correspondantes.empty:
            continue
        if type_colonne == "TIME":
            converties = correspondantes.map(time.fromisoformat)
        else:
            converties = pd.to_datetime(
                correspondantes, format="ISO8601", errors="coerce", utc=avec_fuseau
            )
        yield type_colonne, converties


def inferer_type(
    fichier: str, nom: str, valeurs: pd.Series, format_temporel: tuple[str, str] | None
) -> tuple[str, pd.Series, str, str]:
    """Renvoie le type, les valeurs converties, la provenance et le diagnostic."""
    explicite = "Règle explicite"
    observe = "Valeurs observées"
    nom_normalise = nom.casefold()
    if nom_normalise in IDENTIFIANTS or nom_normalise.endswith(("_id", "_badge")):
        return "IDENTITY", valeurs, explicite, ""
    if fichier == "releves_incidents.csv" and nom in COLONNES_INDICATEURS:
        return "BOOLEAN", valeurs.map(INDICATEURS), explicite, ""
    if format_temporel is not None:
        type_colonne, format_date = format_temporel
        dates = pd.to_datetime(valeurs, format=format_date, errors="coerce")
        return type_colonne, dates, explicite, ""
    if nom in NUMERIQUES.get(fichier, ()):
        return "NUMERIC", convertir_numeriques(valeurs), explicite, ""
    if valeurs.empty:
        return "TEXT", valeurs, observe, "Aucune valeur non vide"

    if valeurs.str.fullmatch(r"[0-9]+").all() and valeurs.str.match(r"0[0-9]").any():
        return "TEXT", valeurs, observe, "Zéros initiaux conservés"

    candidats = [("NUMERIC", convertir_numeriques(valeurs)), *candidats_temporels(valeurs)]
    type_probable, converties = max(candidats, key=lambda candidat: candidat[1].count())
    compatibles = converties.count()
    if compatibles == len(valeurs):
        return type_probable.partition(" ")[0], converties, observe, ""
    if compatibles:
        diagnostic = (
            f"Valeurs mixtes : {compatibles}/{len(valeurs)} compatibles avec {type_probable}"
        )
        return "TEXT", valeurs, observe, diagnostic
    return "TEXT", valeurs, observe, ""


def calculer_statistiques(
    type_colonne: str, valeurs: pd.Series, *, format_date: str | None = None
) -> dict[str, int | float | str]:
    if valeurs.empty:
        return {}
    if type_colonne == "NUMERIC":
        return {
            "min": valeurs.min().item(),
            "max": valeurs.max().item(),
            "mean": valeurs.mean().item(),
            "median": valeurs.median().item(),
        }
    if type_colonne in {"DATE", "TIME", "DATETIME"}:
        bornes = {"min": valeurs.min(), "max": valeurs.max()}
        if format_date is not None:
            return {cle: valeur.strftime(format_date) for cle, valeur in bornes.items()}
        if type_colonne == "DATE":
            return {cle: valeur.date().isoformat() for cle, valeur in bornes.items()}
        return {cle: valeur.isoformat() for cle, valeur in bornes.items()}
    return {}

## Décrire les deux fichiers

Chaque description regroupe les métadonnées, le type des colonnes et les statistiques. Les horaires sont ceux de la machine qui exécute le notebook. La section suivante utilise ces descriptions pour générer les cartes Markdown.

In [33]:
def decrire_csv(chemin: Path) -> dict[str, object]:
    debut = datetime.now().astimezone()
    source = chemin.resolve()
    taille = source.stat().st_size
    donnees, encodage, separateur = lire_csv(source)
    colonnes = []

    for nom, serie in donnees.items():
        # Un champ composé seulement d'espaces compte aussi comme manquant.
        valeurs = serie[serie.notna() & serie.str.strip().ne("")]
        format_temporel = FORMATS_TEMPORELS.get(source.name, {}).get(nom)
        type_colonne, converties, provenance, diagnostic = inferer_type(
            source.name, nom, valeurs, format_temporel
        )
        invalides = int(converties.isna().sum())
        if invalides:
            raise ValueError(
                f"{source.name}, colonne {nom}: {invalides} valeur(s) "
                f"invalide(s) pour le type {type_colonne}."
            )
        colonnes.append({
            "nom": nom,
            "type": type_colonne,
            "provenance": provenance,
            "diagnostic": diagnostic,
            "valeurs_manquantes": len(serie) - len(valeurs),
            "statistiques": calculer_statistiques(
                type_colonne, converties, format_date=format_temporel[1] if format_temporel else None
            ),
        })

    lignes, nombre_colonnes = donnees.shape
    return {
        "source": str(source),
        "taille_octets": taille,
        "debut": debut,
        "fin": datetime.now().astimezone(),
        "encodage": encodage,
        "separateur": separateur,
        "lignes": lignes,
        "nombre_colonnes": nombre_colonnes,
        "colonnes": colonnes,
    }

In [34]:
FICHIERS_CSV = ("releves_incidents.csv", "telemetry.csv")
descriptions = {nom: decrire_csv(DATA_DIR / nom) for nom in FICHIERS_CSV}

pd.DataFrame([
    {
        "Fichier": nom,
        "Encodage": description["encodage"],
        "Séparateur": repr(description["separateur"]),
        "Lignes": description["lignes"],
        "Colonnes": description["nombre_colonnes"],
    }
    for nom, description in descriptions.items()
])

,Fichier,Encodage,Séparateur,Lignes,Colonnes
0,releves_incidents.csv,utf-8,"','",1245,18
1,telemetry.csv,utf-8,"','",135626,7


## Générer les cartes d'identité Markdown

Chaque carte prend le nom du fichier source, la date et l'heure de début (AAAAMMJJ_HHMMSS) de son analyse. Les rapports sont enregistrés dans le dossier daté sous `artifacts/ingestion/identities/`.

In [35]:
IDENTITIES_DIR = Path("artifacts/ingestion/identities")


def formater_taille(octets: int) -> str:
    taille = float(octets)
    for unite in ("octets", "Ko", "Mo", "Go", "To"):
        if taille < 1_000 or unite == "To":
            return f"{taille:.2f} {unite}"
        taille /= 1_000


def rendre_entete_markdown(description: dict[str, object]) -> list[str]:
    source = Path(description["source"])
    debut = description["debut"]
    fin = description["fin"]
    return [
        f"# Carte d'identité — {source.name}",
        "",
        f"- Source: `{source}`",
        f"- Filsize: {formater_taille(description['taille_octets'])}",
        f"- ProcessBeginAt: {debut:%H:%M:%S}",
        f"- ProcessEndAt: {fin:%H:%M:%S}",
        f"- Charset: {description['encodage']}",
        f"- Rows Number: {description['lignes']}",
        f"- Cols Number: {description['nombre_colonnes']}",
    ]


def rendre_tableaux_colonnes(colonnes: list[dict], *, afficher_type_sql: bool = False) -> list[str]:
    niveau = "###" if afficher_type_sql else "##"
    lignes = [
        "",
        f"{niveau} Column list",
        "",
    ]
    if afficher_type_sql:
        lignes.extend([
            "| Colname | SQL Type | Inferred Type | Raison du classement |",
            "| --- | --- | --- | --- |",
        ])
    else:
        lignes.extend([
            "| Colname | Inferred Type | Origine | Diagnostic |",
            "| --- | --- | --- | --- |",
        ])
    statistiques = []
    for colonne in colonnes:
        nom = colonne["nom"].replace("\\", "\\\\").replace("|", "\\|")
        type_colonne = colonne["type"]
        if afficher_type_sql:
            lignes.append(
                f"| {nom} | {colonne['type_sql']} | {type_colonne} | {colonne['raison']} |"
            )
        else:
            provenance = colonne["provenance"]
            diagnostic = colonne["diagnostic"] or "—"
            lignes.append(f"| {nom} | {type_colonne} | {provenance} | {diagnostic} |")
        if type_colonne in {"NUMERIC", "DATE", "TIME", "DATETIME"}:
            valeurs = colonne["statistiques"]
            cellules = [str(valeurs.get(cle, "—")) for cle in ("min", "max", "mean", "median")]
            statistiques.append("| " + " | ".join([nom, *cellules]) + " |")

    if statistiques:
        lignes.extend([
            "",
            f"{niveau} Statistiques",
            "",
            "| Colname | Min value | Max value | Mean value | Median value |",
            "| --- | --- | --- | --- | --- |",
            *statistiques,
        ])
    return lignes


def rendre_carte_markdown(description: dict[str, object]) -> str:
    lignes = rendre_entete_markdown(description) + rendre_tableaux_colonnes(description["colonnes"])
    return "\n".join(lignes) + "\n"


def ecrire_carte_markdown(
    description: dict[str, object], destination: Path, contenu: str
) -> Path:
    """Enregistre une carte sans écraser un rapport existant."""
    debut = description["debut"]
    dossier = destination / f"{debut:%Y%m%d}"
    nom_source = Path(description["source"]).stem
    chemin = dossier / f"{nom_source}_{debut:%Y%m%d_%H%M%S}.md"
    dossier.mkdir(parents=True, exist_ok=True)
    with chemin.open("x", encoding="utf-8", newline="\n") as fichier:
        fichier.write(contenu)
    return chemin

In [36]:
cartes_markdown = {
    nom: ecrire_carte_markdown(description, IDENTITIES_DIR, rendre_carte_markdown(description))
    for nom, description in descriptions.items()
}

for chemin in cartes_markdown.values():
    print(chemin.as_posix())

artifacts/ingestion/identities/20261120/releves_incidents_20261120_152938.md
artifacts/ingestion/identities/20261120/telemetry_20261120_152938.md


## Décrire le script PostgreSQL

SQLGlot lit les définitions de tables et les lignes explicites des `INSERT … VALUES`. La carte indique si le classement vient d'une clé primaire, d'une référence, de la convention de nommage des identifiants ou du type déclaré. Les statistiques décrivent les valeurs présentes dans le script. Les valeurs par défaut calculées par PostgreSQL et les effets des clauses `ON CONFLICT` dépendent de l'exécution en base.

In [37]:
def profil_colonne_sql(colonne: exp.ColumnDef, lignes: list[dict]) -> dict:
    nom = colonne.name
    type_colonne, raison = classer_colonne_sql(colonne)
    valeurs = []
    for ligne in lignes:
        valeur = ligne.get(nom)
        if valeur is None or isinstance(valeur, exp.Null):
            continue
        if isinstance(valeur, exp.Literal):
            valeurs.append(valeur.this)
        elif isinstance(valeur, exp.Neg) and isinstance(valeur.this, exp.Literal):
            valeurs.append(f"-{valeur.this.this}")
        elif type_colonne in {"NUMERIC", "DATE", "TIME", "DATETIME"}:
            raise ValueError(f"Valeur SQL non littérale dans la colonne {nom}.")

    statistiques = {}
    if valeurs and type_colonne == "NUMERIC":
        nombres = convertir_numeriques(pd.Series(valeurs, dtype="string"))
        if nombres.isna().any():
            raise ValueError(f"Valeur numérique SQL invalide dans la colonne {nom}.")
        statistiques = calculer_statistiques(type_colonne, nombres)
    elif valeurs and type_colonne in {"DATE", "TIME", "DATETIME"}:
        if type_colonne == "DATE":
            dates = [date.fromisoformat(valeur) for valeur in valeurs]
        elif type_colonne == "TIME":
            dates = [time.fromisoformat(valeur) for valeur in valeurs]
        else:
            dates = [datetime.fromisoformat(valeur) for valeur in valeurs]
        statistiques = {"min": min(dates).isoformat(), "max": max(dates).isoformat()}

    return {
        "nom": nom,
        "type": type_colonne,
        "type_sql": colonne.kind.sql(dialect=DIALECTE_SQL),
        "raison": raison,
        "statistiques": statistiques,
    }


def decrire_sql(chemin: Path) -> dict:
    debut = datetime.now().astimezone()
    source = chemin.resolve()
    contenu = source.read_bytes()
    encodage = "utf-8-sig" if contenu.startswith(BOM_UTF8) else "utf-8"
    try:
        instructions = [
            instruction for instruction in parse(contenu.decode(encodage), read=DIALECTE_SQL)
            if instruction is not None
        ]
        tables = extraire_tables_sql(instructions)
        profils = [
            {
                "nom": nom,
                "lignes": len(table["lignes"]),
                "colonnes": [
                    profil_colonne_sql(colonne, table["lignes"])
                    for colonne in table["definitions"].values()
                ],
            }
            for nom, table in tables.items()
        ]
    except (ValueError, TokenError, ParseError) as erreur:
        raise ValueError(f"{source.name}: {erreur}") from erreur
    return {
        "source": str(source),
        "taille_octets": len(contenu),
        "debut": debut,
        "fin": datetime.now().astimezone(),
        "encodage": encodage,
        "instructions": len(instructions),
        "tables": profils,
        "lignes": sum(table["lignes"] for table in profils),
        "nombre_colonnes": sum(len(table["colonnes"]) for table in profils),
    }

In [38]:
def rendre_carte_sql(description: dict) -> str:
    lignes = rendre_entete_markdown(description)
    lignes.extend([
        "- Dialecte: PostgreSQL",
        f"- Instructions: {description['instructions']}",
        f"- Tables: {len(description['tables'])}",
        "",
        "Les lignes comptées correspondent aux tuples des INSERT. Les statistiques "
        "portent sur les valeurs explicites. Le symbole — indique une statistique "
        "indisponible ou sans objet, notamment pour les champs alimentés par DEFAULT NOW().",
    ])
    for table in description["tables"]:
        lignes.extend([
            "",
            f"## Table `{table['nom']}`",
            "",
            f"- Rows Number: {table['lignes']}",
            f"- Cols Number: {len(table['colonnes'])}",
            *rendre_tableaux_colonnes(table["colonnes"], afficher_type_sql=True),
        ])
    return "\n".join(lignes) + "\n"


description_sql = decrire_sql(DATA_DIR / "machine.sql.txt")
carte_sql = ecrire_carte_markdown(
    description_sql, IDENTITIES_DIR, rendre_carte_sql(description_sql)
)
print(carte_sql.as_posix())

artifacts/ingestion/identities/20261120/machine.sql_20261120_152950.md


# Étape 2 — Charger les sources dans Bronze

Les deux CSV et le script PostgreSQL alimentent quatre tables, car `machine.sql.txt` contient les données de `machine` et de `maintenance`. Le chargeur Python prépare les lignes, SQLGlot lit les `INSERT` sans exécuter le script, puis `psql` utilise `COPY` dans PostgreSQL. Chaque chargement remplace son jeu de données dans une transaction, ce qui permet de rejouer l'opération sans ajouter de lignes.

| Choix | Pourquoi |
| --- | --- |
| Valeurs sources en texte, suivies de leur colonne `_type` | Préserver la représentation d'origine, dont les zéros initiaux, tout en gardant le type métier visible. |
| Une colonne `hash` par ligne | Vérifier qu'un rejeu avec la même source et la même clé retrouve les mêmes empreintes. Le HMAC-SHA-256 porte sur les valeurs sources avant pseudonymisation. |
| Nom et badge pseudonymisés séparément par SHA-256 salé | Retirer les identifiants d'opérateur en clair de ces deux colonnes et préserver les associations observées. Le sel et la clé restent hors du dépôt. |
| Valeurs SQL non fournies conservées à `NULL` | Les champs remplis par un `DEFAULT` ne reçoivent pas de valeur inventée à l'ingestion. Une chaîne vide reste distincte de `NULL`. |

## 1. Démarrer PostgreSQL et préparer les secrets

Les cellules suivantes exécutent le chargement Bronze dans l’ordre. Docker Desktop doit être lancé et le fichier `docker-compose.yml` se trouve dans le dossier parent. Au premier lancement, le notebook crée le fichier local `.env.bronze`. Les exécutions suivantes réutilisent ses secrets, sans les afficher.

`CHARGER_BRONZE = False` laisse les tables en place et exécute seulement l’aperçu final. Pour recharger, utilise **Tout exécuter** ou relance les cellules Bronze depuis le début.

In [10]:
from inspect import getsource
from itertools import chain
from secrets import token_hex
from subprocess import CalledProcessError, run
from tempfile import TemporaryFile

from IPython.display import Code, display

from bronze import (
    charger_psql,
    commande_psql,
    encoder_champs,
    lire_csv_bronze,
    lire_secrets,
    lire_tables_bronze,
    lire_tables_machine,
    preparer_tables,
    transformer_ligne,
)


CHARGER_BRONZE = True
COMPOSE_BRONZE = Path("../docker-compose.yml")
SECRETS_BRONZE = Path(".env.bronze")


def preparer_secrets_bronze(chemin: Path) -> None:
    try:
        with chemin.open("x", encoding="ascii") as fichier:
            fichier.write(f"BRONZE_OPERATOR_SALT={token_hex(32)}\n")
            fichier.write(f"BRONZE_ROW_HASH_KEY={token_hex(32)}\n")
    except FileExistsError:
        # Réutiliser les mêmes valeurs assure la stabilité des empreintes.
        pass


if CHARGER_BRONZE:
    run(
        ["docker", "compose", "-f", str(COMPOSE_BRONZE), "up", "-d", "--wait",
         "--wait-timeout", "60", "db"],
        check=True,
    )
    preparer_secrets_bronze(SECRETS_BRONZE)
    variables_bronze = dict(
        ligne.split("=", 1)
        for ligne in SECRETS_BRONZE.read_text(encoding="utf-8-sig").splitlines()
        if "=" in ligne and not ligne.lstrip().startswith("#")
    )
    cle_bronze, sel_bronze = lire_secrets("releves_incidents", variables_bronze)
    print("PostgreSQL est prêt. Les secrets restent dans .env.bronze.")

PostgreSQL est prêt. Les secrets restent dans .env.bronze.


## 2. Lire les sources

Les CSV sont lus en texte ligne par ligne. SQLGlot associe les valeurs littérales des `INSERT` aux colonnes du `CREATE TABLE`, dans l’ordre du schéma. Un champ absent ou un `NULL` SQL est représenté par `None`. La première ligne lue est conservée dans le flux qui sera chargé.

In [11]:
sources_bronze = {
    "releves_incidents": DATA_DIR / "releves_incidents.csv",
    "telemetry": DATA_DIR / "telemetry.csv",
    "machine": DATA_DIR / "machine.sql.txt",
}

display(Code(getsource(lire_csv_bronze) + "\n" + getsource(lire_tables_machine), language="python"))

if CHARGER_BRONZE:
    tables_sources = []
    for dataset, source in sources_bronze.items():
        for nom, types, lignes in lire_tables_bronze(source, dataset):
            lignes = iter(lignes)
            premiere = next(lignes, None)
            if premiere is None:
                raise ValueError(f"{source.name} : la table {nom} est vide.")
            tables_sources.append((nom, types, premiere, chain([premiere], lignes)))
    display(pd.DataFrame([
        {"Table": nom, "Colonnes sources": len(types), "Colonnes dans l’ordre": ", ".join(types)}
        for nom, types, _, _ in tables_sources
    ]))

def lire_csv_bronze(source: Path, types: dict[str, str]) -> Iterator[list[str]]:
    """Conserve les champs CSV sous forme de texte dans leur ordre d'origine."""
    with source.open(encoding="utf-8-sig", newline="") as fichier:
        lecteur = csv.reader(fichier, strict=True)
        if next(lecteur, None) != list(types):
            raise ValueError(f"{source.name} : en-tête différent du contrat Bronze.")
        for ligne in lecteur:
            if len(ligne) != len(types) or any("\x00" in valeur for valeur in ligne):
                raise ValueError(f"{source.name} : ligne CSV invalide à la ligne {lecteur.line_num}.")
            yield ligne

def lire_tables_machine(source: Path) -> Iterator[tuple[str, dict[str, str], list[list[Valeur]]]]:
    """Lit les littéraux des INSERT dans l’ordre du schéma, sans exécuter le dump."""
    try:
        instructions = [i for i in parse(source.read_text(encoding="utf-8-sig"), read=DIALECTE_SQL) if i]
        tables = extraire_tables_sql(instructions)
        if tables.keys() != set(TABLES_MACHINE):
            raise ValueError("les tables machine et maintenance sont attendues.")
        for nom in TABLES_MACHINE:
            table = tables[nom]
            types = {col: classer_colonne_sql(d)[0] for col, d in table["definitions"].items()}
            lignes = [
                [valeur_sql(ligne.get(col), f"{nom}.{col}") for col in types]
                for ligne in table["lignes"]
            ]
            yield nom, types, lignes
    except (ValueError, ParseError, TokenError) as erreur:
        raise ValueError(f"{source.name} : {erreur}") from erreur

,Table,Colonnes sources,Colonnes dans l’ordre
0,releves_incidents,18,"incident_id, date, time, operator_name, machin..."
1,telemetry,7,"machine_id, timestamp, temperature_c, pressure..."
2,machine,11,"machine_code, commissioning_date, max_daily_ca..."
3,maintenance,11,"maintenance_id, machine_code, maintenance_at, ..."


## 3. Fabriquer les valeurs Bronze

Le code affiché ci-dessous est celui utilisé pour chaque ligne. Il calcule le HMAC sur les valeurs d’origine encodées en JSON, remplace les champs d’opérateur par leurs SHA‑256 salés et ajoute après chaque valeur son type. Le tableau montre une ligne par table. La valeur source du badge est masquée et les empreintes sont raccourcies à l’affichage.

In [12]:
display(Code(getsource(encoder_champs) + "\n" + getsource(transformer_ligne), language="python"))

champs_demo = {
    "releves_incidents": "operator_badge",
    "telemetry": "temperature_c",
    "machine": "created_at",
    "maintenance": "duration_hours",
}


def enrichir_table_bronze(lignes, types, sel_operateur):
    for ligne in lignes:
        yield transformer_ligne(ligne, types, cle_bronze, sel_operateur=sel_operateur)


if CHARGER_BRONZE:
    tables_preparees = []
    exemples_prepares = []
    for nom, types, premiere_source, lignes in tables_sources:
        sel_operateur = sel_bronze if nom == "releves_incidents" else None
        preparees = enrichir_table_bronze(lignes, types, sel_operateur)
        premiere_bronze = next(preparees)
        champ = champs_demo[nom]
        index = list(types).index(champ)
        valeur_source = "[masquée]" if sel_operateur else premiere_source[index]
        valeur_bronze = premiere_bronze[2 * index]
        exemples_prepares.append({
            "Table": nom,
            "Champ": champ,
            "Source": valeur_source,
            "Bronze": valeur_bronze[:12] + "…" if sel_operateur and valeur_bronze else valeur_bronze,
            "_type": premiere_bronze[2 * index + 1],
            "hash": premiere_bronze[-1][:12] + "…",
        })
        tables_preparees.append((nom, types, chain([premiere_bronze], preparees)))
    display(pd.DataFrame(exemples_prepares))

def encoder_champs(champs: Sequence[Valeur]) -> bytes:
    """Encode une séquence ordonnée sans ambiguïté liée aux séparateurs."""
    return json.dumps(champs, ensure_ascii=False, separators=(",", ":")).encode("utf-8")

def transformer_ligne(
    ligne: Sequence[Valeur], types: dict[str, str], cle_hash: bytes,
    *, sel_operateur: bytes | None = None,
) -> list[Valeur]:
    """Ajoute les types et l’empreinte et pseudonymise si la source le demande."""
    empreinte = hmac.digest(cle_hash, encoder_champs(ligne), "sha256").hex()
    resultat = []
    for (colonne, type_colonne), valeur in zip(types.items(), ligne, strict=True):
        if sel_operateur is not None and colonne in CHAMPS_OPERATEUR and valeur:
            # Sépare les domaines nom et badge, même si une valeur se répète.
            valeur = hashlib.sha256(sel_operateur + b"\0" + encoder_champs([colonne, valeur])).hexdigest()
        resultat.extend((valeur, type_colonne))
    return [*resultat, empreinte]

,Table,Champ,Source,Bronze,_type,hash
0,releves_incidents,operator_badge,[masquée],df3e1bdfdf46…,IDENTITY,5dffdb6103ce…
1,telemetry,temperature_c,46.348,46.348,NUMERIC,45d2e441c188…
2,machine,created_at,NaN,NaN,DATETIME,1e8c54e29b6f…
3,maintenance,duration_hours,2.24,2.24,NUMERIC,b61abd81b203…


## 4. Écrire les lignes dans PostgreSQL

Les lignes enrichies sont préparées au format `COPY`. Une seule transaction remplace les quatre tables Bronze. Si la préparation échoue, la commande SQL n’est pas envoyée. Si un chargement échoue en base, la transaction est annulée.

In [13]:
display(Code(getsource(preparer_tables), language="python"))

if CHARGER_BRONZE:
    with TemporaryFile() as flux:
        volumes_charges = preparer_tables(flux, tables_preparees)
        charger_psql(COMPOSE_BRONZE, flux)
    display(pd.DataFrame([
        {"Table": nom, "Lignes chargées": total} for nom, total in volumes_charges.items()
    ]))
else:
    print("Chargement ignoré. L’aperçu ci-dessous consulte les tables existantes.")

def preparer_tables(flux: BinaryIO, tables: Iterable[TableBronze]) -> dict[str, int]:
    """Prépare une transaction COPY à partir de lignes déjà enrichies."""
    flux.write(b'BEGIN;\nCREATE SCHEMA IF NOT EXISTS "bronze";\n')
    volumes = {
        nom: ecrire_table_copy(flux, nom, types, lignes, autoriser_null=nom in TABLES_MACHINE)
        for nom, types, lignes in tables
    }
    flux.write(b"COMMIT;\n")
    flux.seek(0)
    return volumes

,Table,Lignes chargées
0,releves_incidents,1245
1,telemetry,135626
2,machine,15
3,maintenance,115


## 5. Vérifier ce qui se trouve dans PostgreSQL

Cette dernière cellule compte les lignes et les colonnes directement en base. Elle affiche aussi un exemple de valeur stockée, son type et son empreinte pour chaque table.

In [14]:
TABLES_BRONZE = ("releves_incidents", "telemetry", "machine", "maintenance")


def lire_bronze(requete: str) -> pd.DataFrame:
    """Lit le résultat d'une requête dans le PostgreSQL local."""
    sql = f"BEGIN READ ONLY;\nCOPY ({requete}) TO STDOUT WITH (FORMAT CSV, HEADER);\nCOMMIT;\n"
    resultat = run(
        commande_psql(COMPOSE_BRONZE, "-q"),
        input=sql.encode("utf-8"),
        capture_output=True,
        check=True,
    )
    return pd.read_csv(StringIO(resultat.stdout.decode("utf-8")), keep_default_na=False)


comptages = "\nUNION ALL\n".join(
    f"SELECT '{nom}' AS nom_table, count(*) AS lignes FROM bronze.{nom}"
    for nom in TABLES_BRONZE
)
requete_volumes = f"""
WITH volumes AS ({comptages})
SELECT nom_table AS "Table", lignes AS "Lignes", count(c.column_name) AS "Colonnes"
FROM volumes
JOIN information_schema.columns AS c
    ON c.table_schema = 'bronze' AND c.table_name = nom_table
GROUP BY nom_table, lignes
ORDER BY nom_table
"""

requete_exemples = """
(SELECT 'releves_incidents' AS "Table", incident_id AS "Identifiant",
        'operator_badge' AS "Champ", left(operator_badge, 12) || '…' AS "Valeur",
        operator_badge_type AS "Type", left(hash, 12) || '…' AS "Hash"
 FROM bronze.releves_incidents ORDER BY incident_id LIMIT 1)
UNION ALL
(SELECT 'telemetry', machine_id, 'temperature_c', temperature_c, temperature_c_type,
        left(hash, 12) || '…'
 FROM bronze.telemetry WHERE temperature_c <> ''
 ORDER BY "timestamp", machine_id, hash LIMIT 1)
UNION ALL
(SELECT 'machine', machine_code, 'max_daily_capacity', max_daily_capacity,
        max_daily_capacity_type, left(hash, 12) || '…'
 FROM bronze.machine ORDER BY machine_code LIMIT 1)
UNION ALL
(SELECT 'maintenance', maintenance_id, 'duration_hours', duration_hours,
        duration_hours_type, left(hash, 12) || '…'
 FROM bronze.maintenance ORDER BY length(maintenance_id), maintenance_id LIMIT 1)
"""

try:
    volumes_bronze = lire_bronze(requete_volumes)
    exemples_bronze = lire_bronze(requete_exemples)
except (OSError, CalledProcessError) as erreur:
    print("Aperçu Bronze indisponible. Exécute la cellule de chargement ci-dessus.")
    detail = getattr(erreur, "stderr", None)
    if detail:
        print(detail.decode("utf-8", errors="replace").strip())
else:
    display(volumes_bronze, exemples_bronze)

,Table,Lignes,Colonnes
0,machine,15,23
1,maintenance,115,23
2,releves_incidents,1245,37
3,telemetry,135626,15


,Table,Identifiant,Champ,Valeur,Type,Hash
0,releves_incidents,INC-000001,operator_badge,df3e1bdfdf46…,IDENTITY,5dffdb6103ce…
1,telemetry,MACH-01,temperature_c,46.332,NUMERIC,32204f1643b0…
2,machine,MACH-01,max_daily_capacity,770,NUMERIC,1e8c54e29b6f…
3,maintenance,1,duration_hours,2.24,NUMERIC,b61abd81b203…
